# BiLSTM Hyperparameter Tuning

Search over the BiLSTM configurations used for the distraction prediction model and save the best checkpoint.

In [1]:
import json
import sys
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        marker = candidate / "distraction_prediction" / "models" / "lstm_model.py"
        if marker.exists():
            return candidate
    raise FileNotFoundError("Could not locate the project root.")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from distraction_prediction.models.lstm_model import DistractionLSTM

WINDOWS_DIR = PROJECT_ROOT / "distraction_prediction" / "data" / "processed" / "windows"
SAVE_DIR = PROJECT_ROOT / "distraction_prediction" / "models" / "saved_models"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [2]:
X_train = np.load(WINDOWS_DIR / "X_train.npy")
y_train = np.load(WINDOWS_DIR / "y_train.npy")
X_val = np.load(WINDOWS_DIR / "X_val.npy")
y_val = np.load(WINDOWS_DIR / "y_val.npy")

print(f"Train: {X_train.shape}, distracted rate={y_train.mean():.3f}")
print(f"Val  : {X_val.shape}, distracted rate={y_val.mean():.3f}")

Train: (51371, 10, 24), distracted rate=0.574
Val  : (6421, 10, 24), distracted rate=0.574


In [3]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, total_correct, total_items = 0.0, 0, 0
    for features, labels in loader:
        features, labels = features.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(features).squeeze()
        loss = criterion(logits, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item() * len(labels)
        predictions = (torch.sigmoid(logits) >= 0.5).long()
        total_correct += (predictions == labels.long()).sum().item()
        total_items += len(labels)

    return total_loss / total_items, total_correct / total_items


def evaluate_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, total_correct, total_items = 0.0, 0, 0
    with torch.no_grad():
        for features, labels in loader:
            features, labels = features.to(device), labels.to(device)
            logits = model(features).squeeze()
            loss = criterion(logits, labels)

            total_loss += loss.item() * len(labels)
            predictions = (torch.sigmoid(logits) >= 0.5).long()
            total_correct += (predictions == labels.long()).sum().item()
            total_items += len(labels)

    return total_loss / total_items, total_correct / total_items

In [4]:
configs = [
    {"hidden_size": 64, "num_layers": 1, "dropout": 0.5, "lr": 5e-4, "weight_decay": 1e-3, "batch_size": 256},
    {"hidden_size": 32, "num_layers": 1, "dropout": 0.5, "lr": 5e-4, "weight_decay": 1e-3, "batch_size": 256},
    {"hidden_size": 64, "num_layers": 2, "dropout": 0.5, "lr": 5e-4, "weight_decay": 5e-3, "batch_size": 256},
    {"hidden_size": 128, "num_layers": 1, "dropout": 0.6, "lr": 3e-4, "weight_decay": 1e-3, "batch_size": 512},
    {"hidden_size": 64, "num_layers": 1, "dropout": 0.4, "lr": 1e-3, "weight_decay": 1e-2, "batch_size": 128},
]

pos_weight = torch.tensor([(len(y_train) - y_train.sum()) / y_train.sum()], dtype=torch.float32, device=device)
X_train_tensor = torch.FloatTensor(X_train)
y_train_tensor = torch.FloatTensor(y_train)
X_val_tensor = torch.FloatTensor(X_val)
y_val_tensor = torch.FloatTensor(y_val)

best_val_loss = float("inf")
best_val_acc = 0.0
best_config = None
best_state = None
results = []

for index, config in enumerate(configs, start=1):
    print(f"Config {index}/{len(configs)}: {config}")

    train_loader = DataLoader(
        TensorDataset(X_train_tensor, y_train_tensor),
        batch_size=config["batch_size"],
        shuffle=True,
    )
    val_loader = DataLoader(
        TensorDataset(X_val_tensor, y_val_tensor),
        batch_size=config["batch_size"],
    )

    model = DistractionLSTM(
        input_size=X_train.shape[2],
        hidden_size=config["hidden_size"],
        num_layers=config["num_layers"],
        dropout=config["dropout"],
        head_name="classifier",
    ).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config["lr"], weight_decay=config["weight_decay"])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=5)

    config_best_loss = float("inf")
    config_best_acc = 0.0
    config_best_epoch = 0
    config_best_state = None
    epochs_without_improvement = 0
    start_time = time.time()

    for epoch in range(1, 101):
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc = evaluate_epoch(model, val_loader, criterion, device)
        scheduler.step(val_loss)

        if val_loss < config_best_loss:
            config_best_loss = val_loss
            config_best_acc = val_acc
            config_best_epoch = epoch
            config_best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= 15:
            break

    elapsed = time.time() - start_time
    print(
        f"  Best epoch {config_best_epoch}: val_loss={config_best_loss:.4f}, "
        f"val_acc={config_best_acc:.4f}, time={elapsed:.0f}s"
    )

    results.append({
        "config": config,
        "val_loss": config_best_loss,
        "val_acc": config_best_acc,
        "best_epoch": config_best_epoch,
        "time_seconds": round(elapsed, 2),
    })

    if config_best_loss < best_val_loss:
        best_val_loss = config_best_loss
        best_val_acc = config_best_acc
        best_config = config
        best_state = config_best_state

print("\nBest config:", best_config)
print(f"Best validation loss: {best_val_loss:.4f}")
print(f"Best validation accuracy: {best_val_acc:.4f}")

Config 1/5: {'hidden_size': 64, 'num_layers': 1, 'dropout': 0.5, 'lr': 0.0005, 'weight_decay': 0.001, 'batch_size': 256}
  Best epoch 48: val_loss=0.2611, val_acc=0.8728, time=149s
Config 2/5: {'hidden_size': 32, 'num_layers': 1, 'dropout': 0.5, 'lr': 0.0005, 'weight_decay': 0.001, 'batch_size': 256}
  Best epoch 76: val_loss=0.2727, val_acc=0.8542, time=240s
Config 3/5: {'hidden_size': 64, 'num_layers': 2, 'dropout': 0.5, 'lr': 0.0005, 'weight_decay': 0.005, 'batch_size': 256}
  Best epoch 75: val_loss=0.2034, val_acc=0.9161, time=247s
Config 4/5: {'hidden_size': 128, 'num_layers': 1, 'dropout': 0.6, 'lr': 0.0003, 'weight_decay': 0.001, 'batch_size': 512}
  Best epoch 71: val_loss=0.2474, val_acc=0.8788, time=169s
Config 5/5: {'hidden_size': 64, 'num_layers': 1, 'dropout': 0.4, 'lr': 0.001, 'weight_decay': 0.01, 'batch_size': 128}
  Best epoch 29: val_loss=0.2322, val_acc=0.8919, time=153s

Best config: {'hidden_size': 64, 'num_layers': 2, 'dropout': 0.5, 'lr': 0.0005, 'weight_decay':

In [5]:
model_path = SAVE_DIR / "best_model_finetuned.pt"
results_path = SAVE_DIR / "bilstm_finetuned_results.json"

torch.save(
    {
        "model_state_dict": best_state,
        "val_loss": best_val_loss,
        "config": {
            "input_size": X_train.shape[2],
            "hidden_size": best_config["hidden_size"],
            "num_layers": best_config["num_layers"],
            "dropout": best_config["dropout"],
            "bidirectional": True,
            "attention_activation": "tanh",
            "head_name": "classifier",
        },
    },
    model_path,
)

with open(results_path, "w", encoding="utf-8") as handle:
    json.dump(
        {
            "val_loss": round(best_val_loss, 4),
            "val_acc": round(best_val_acc, 4),
            "best_config": best_config,
            "all_results": results,
        },
        handle,
        indent=2,
    )

print(f"Saved model: {model_path}")
print(f"Saved results: {results_path}")

Saved model: E:\SDPPS\distraction_prediction\models\saved_models\best_model_finetuned.pt
Saved results: E:\SDPPS\distraction_prediction\models\saved_models\bilstm_finetuned_results.json
